# Week 5 Semantic Search Evaluation

This notebook reviews embedding-based listing search on a fixed 10k sample, then compares MiniLM, MPNet, and BM25 keyword retrieval.


In [1]:
import sys
import time

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)

sys.path.append("..")

from src.real_estate_nlp.keyword_search import BM25Searcher
from src.real_estate_nlp.semantic_search import SemanticSearcher

---

## 1. Artifacts Loading

The 10k sample is the working set for latency checks and retrieval review.

In [2]:
sample = pd.read_csv("../data/processed/listing_semantic_sample_10k.csv")
sample.shape

(10000, 9)

In [3]:
sample[["listing_id", "city", "price", "beds", "baths", "sqft", "remarks_cleaned"]].head(3)

,listing_id,city,price,beds,baths,sqft,remarks_cleaned
0,1149776480,Irvine,899000,2.0,2.0,1260.0,"luxury top floor penthouse life style with panoramic pool views at watermarke, irvine. welcome this top floor 2 bedroom, 2 bath condo offers total privacy with no neighbors above, no neighbor next door, and unobstructed views of the resort style pool, this residence blends sophistication, comfort, and exclusivity. the home welcomes you with a bright, expansive open layout and an upgraded gourmet kitchen appointed with granite countertops, stainless steel appliances, and elegant custom cabinetry. the primary suite provides a tranquil retreat with serene pool vistas, a spacious walk in closet, and a luxuriously appointed en suite bath. the secondary bedroom is equally well designed, featuring generous space and storage. a noteworthy highlight of this 1260 square feet of spacious living property is the inclusion of 2 private, covered parking spaces located on the same level, 4th floor a rare and highly desirable convenience within the community. an in unit laundry room adds to the overall ease and functionality of the home. residents enjoy access to watermarke's extensive amenity collection, including a 7 days a week concierge, a grand clubhouse with movie theater and business center, a junior olympic sized heated pool, private gated area, multiple spas, a state of the art fitness center, tennis and basketball courts, and beautifully landscaped grounds. situated moments from uc irvine, john wayne airport, irvine spectrum, premier shopping, dining, and irvine's finest conveniences, this penthouse offers an exceptional blend of luxury and accessibility."
1,1150462921,Menifee,630000,4.0,2.0,1982.0,"welcome home to stunning views and 1 story living in menifee enjoy breathtaking valley views and unforgettable sunsets from this highly upgraded 4 bedroom, 2 bath, 1 story home located in the heart of menifee. inside, you'll find an open concept floor plan featuring upgraded flooring, elegant arched doorways, vaulted ceilings, and floor to ceiling windows that flood the home with natural light while showcasing the spectacular views. the formal living room flows effortlessly into the main living space, where the kitchen shines with a raised center island breakfast bar, gleaming tile countertops, ample cabinetry, and pull out drawers for added functionality.the kitchen opens to the dining area and family room, complete with a cozy fireplace and direct access to the backyard perfect for entertaining. the spacious primary suite offers private backyard access and a luxurious ensuite with dual sink vanities, a separate soaking tub and shower, and a large walk in closet. three additional bedrooms are generously sized, ideal for family, guests, or a home office.step outside to your private oasis featuring lush, mature landscaping, rock accents, a cement retaining wall, drip irrigation system, and vinyl fencing. relax under the expansive covered patio with built in lighting and an automatic electric shade, the perfect spot to unwind while enjoying menifee's stunning sunsets.additional highlights include plantation shutters throughout, art accent lighting, inside laundry, a 3 car garage, this is a rare opportunity to own a beautifully upgraded home with incredible views hurry, this one won't last"
2,1168736420,Long Barn,524900,3.0,2.0,1985.0,"what a fun place to live full time or go spend quality time . this is a move in ready mountain retreat in the special gated sierra park community a short drice to pinecrest lake and dodge ridge ski area. situated on a desirable double lot, this custom 1985 square feet home offers 3 bedroom, 2 bathroom, a deep oversized 2 car garage, carport, and composition roof. just 15 miles from pinecrest lake and dodge ridge ski resort, with sierra park amenities including a private swimming lake, playground, and lodge. the spacious great room features a high coffered ceiling, cozy fireplace, skylights, sun tubes, and abundant

---

## 2. Search Indexes

The two semantic searchers load saved FAISS indexes built from the same 10k records. BM25 uses the same text as a keyword baseline.

In [4]:
records = sample.to_dict("records")

minilm = SemanticSearcher(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    local_files_only=True,
    batch_size=8,
).load("../data/models/semantic/sentence-transformers_all-MiniLM-L6-v2", "sample_10k")

mpnet = SemanticSearcher(
    model_name="sentence-transformers/all-mpnet-base-v2",
    local_files_only=True,
    batch_size=4,
).load("../data/models/semantic/sentence-transformers_all-mpnet-base-v2", "sample_10k")

bm25 = BM25Searcher().build(records)

embedding_searchers = {"MiniLM": minilm, "MPNet": mpnet}
search_methods = {"MiniLM": minilm, "MPNet": mpnet, "BM25": bm25}

pd.DataFrame([
    {"method": name, "listings": len(searcher.metadata), "embedding_dim": searcher.embeddings.shape[1]}
    for name, searcher in embedding_searchers.items()
])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,method,listings,embedding_dim
0,MiniLM,10000,384
1,MPNet,10000,768


Two models are used and compared in this study.

- `all-MiniLM-L6-v2` (default model)
  - small and fast
  - 384-dimensional
- `all-mpnet-base-v2`
  - heavy
  - 768-dimensional

---

## 3. Retrieval Examples

We first check whether the top results read like a reasonable answer to the query.

In [5]:
query = "home with a private pool and outdoor entertaining space"

def compact_results(results):
    cols = ["rank", "score", "listing_id", "city", "price", "beds", "baths", "text"]
    df = pd.DataFrame(results)
    df["text"] = df["search_text"]
    return df[cols]

example_results = []
for method, searcher in search_methods.items():
    result = compact_results(searcher.search(query, top_k=3))
    result.insert(0, "method", method)
    example_results.append(result)

pd.concat(example_results, ignore_index=True)

,method,rank,score,listing_id,city,price,beds,baths,text
0,MiniLM,1,0.737546,1111896555,Fontana,625000,4.0,2.0,"large 4 bedroom 2 bathroom home with living room dining off kitchen. enclosed patio 300 square feet pool. close to schools, shopping freeways."
1,MiniLM,2,0.733873,1159158818,Blythe,245000,3.0,2.0,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
2,MiniLM,3,0.696034,1158583082,Palmdale,479900,4.0,2.0,"ready for your high efficiency pool home here it is palmdale pool home with 4 bedroom and 2 bathroom. this home features triple pane windows, new tankless water heater, upgraded electrical panel, foam insulated walls, and zone heating and cooling. 2 evaporative coolers and central heat, plus gas stove. whole home water filtration system. living room has vaulted wood ceilings with large windows that create a bright open space. remodeled kitchen with new stainless steel appliances. spacious master bedroom with 2 closets 1 walk in . remodeled en suite master bathroom. 3 secondary bedrooms. remodeled hall bath with jacuzzi tub. new carpet, faux wood window covers, and paint throughout. the private backyard is a great entertaining space with 2 covered patio areas. enjoy the pebble tech in ground pool, with all upgraded plumbing and newer equipment. fully fenced for safety."
3,MPNet,1,0.763622,1159158818,Blythe,245000,3.0,2.0,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
4,MPNet,2,0.738082,1156477613,Riverside,865000,4.0,4.0,"location location welcome to your private pool spa home. in orangecrest experience the best of southern california living in this beautifully upgraded home, where a resort style backyard and thoughtfully designed interior come together for the ultimate in comfort and entertaining. from the moment you arrive, stunning manicured curb app

---

## 4. Comparison Query Set

These queries cover common real estate search patterns: amenities, views, condition, layout, outdoor space, and location language.

In [6]:
eval_queries = [
    {"query": "home with a private pool and outdoor entertaining space", "terms": ["pool", "outdoor", "patio", "backyard"]},
    {"query": "ocean view home near the beach", "terms": ["ocean", "beach", "coast", "water view"]},
    {"query": "updated kitchen with modern finishes", "terms": ["updated kitchen", "remodeled kitchen", "modern", "stainless"]},
    {"query": "single story home with open floor plan", "terms": ["single story", "one story", "open floor"]},
    {"query": "family home close to parks and schools", "terms": ["school", "park", "family"]},
    {"query": "condo with resort style amenities", "terms": ["condo", "resort", "clubhouse", "fitness"]},
    {"query": "large lot with room for expansion", "terms": ["large lot", "expansion", "acre", "room to"]},
    {"query": "move in ready home with upgrades", "terms": ["move in ready", "turnkey", "upgraded", "updated"]},
    {"query": "home office or flexible bonus room", "terms": ["office", "bonus room", "den", "flex"]},
    {"query": "garage parking and extra storage", "terms": ["garage", "parking", "storage"]},
]

pd.DataFrame(eval_queries)

,query,terms
0,home with a private pool and outdoor entertaining space,"[pool, outdoor, patio, backyard]"
1,ocean view home near the beach,"[ocean, beach, coast, water view]"
2,updated kitchen with modern finishes,"[updated kitchen, remodeled kitchen, modern, stainless]"
3,single story home with open floor plan,"[single story, one story, open floor]"
4,family home close to parks and schools,"[school, park, family]"
5,condo with resort style amenities,"[condo, resort, clubhouse, fitness]"
6,large lot with room for expansion,"[large lot, expansion, acre, room to]"
7,move in ready home with upgrades,"[move in ready, turnkey, upgraded, updated]"
8,home office or flexible bonus room,"[office, bonus room, den, flex]"
9,garage parking and extra storage,"[garage, parking, storage]"


---

## 5. Latency Study

This section measures search latency on the fixed 10k listing sample.

Two timings are useful:

- End-to-end latency: the practical search time, including query embedding and retrieval.
- FAISS-only latency: a diagnostic lookup time, using already-computed query embeddings against the 10k-vector index.

In [7]:
def milliseconds(values):
    return [round(value * 1000, 2) for value in values]


timing_rows = []

for method, searcher in search_methods.items():
    times = []
    for item in eval_queries:
        start = time.perf_counter()
        searcher.search(item["query"], top_k=10)
        times.append(time.perf_counter() - start)

    timing_rows.append({
        "method": f"{method}_end_to_end",
        "avg_ms": np.mean(milliseconds(times)),
        "p95_ms": np.percentile(milliseconds(times), 95),
    })

for method, searcher in embedding_searchers.items():
    query_embeddings = searcher.encode_queries([item["query"] for item in eval_queries])
    times = []
    for query_embedding in query_embeddings:
        start = time.perf_counter()
        searcher.index.search(query_embedding.reshape(1, -1), 10)
        times.append(time.perf_counter() - start)

    timing_rows.append({
        "method": f"{method}_faiss_lookup_only",
        "avg_ms": np.mean(milliseconds(times)),
        "p95_ms": np.percentile(milliseconds(times), 95),
    })

pd.DataFrame(timing_rows)

,method,avg_ms,p95_ms
0,MiniLM_end_to_end,8.655,17.7905
1,MPNet_end_to_end,13.778,26.5575
2,BM25_end_to_end,11.338,15.5705
3,MiniLM_faiss_lookup_only,0.226,0.2555
4,MPNet_faiss_lookup_only,0.469,0.5355


---

## 6. Retrieval Quality Evaluation

This section uses the same 10-query set to compare top-5 retrieval quality across MiniLM, MPNet, and BM25.

- Precision@5 is kept as the compact comparison metric.
- The detailed review table omits row-level True/False relevance flags so the output stays readable.
- The score is still a lightweight keyword-based estimate, so it should be read as a quick check rather than final human judgment.

In [8]:
def relevant(result, terms):
    text = f"{result.get('city', '')} {result.get('search_text', '')}".lower()
    return any(term in text for term in terms)


review_rows = []

for item in eval_queries:
    for method, searcher in search_methods.items():
        for result in searcher.search(item["query"], top_k=5):
            review_rows.append({
                "method": method,
                "query": item["query"],
                "rank": result["rank"],
                "listing_id": result["listing_id"],
                "city": result["city"],
                "score": result["score"],
                "remarks": result["search_text"],
                "_relevant": relevant(result, item["terms"]),
            })

review_df = pd.DataFrame(review_rows)
review_df[["method", "query", "rank", "listing_id", "city", "score", "remarks"]].head(10)

,method,query,rank,listing_id,city,score,remarks
0,MiniLM,home with a private pool and outdoor entertaining space,1,1111896555,Fontana,0.737546,"large 4 bedroom 2 bathroom home with living room dining off kitchen. enclosed patio 300 square feet pool. close to schools, shopping freeways."
1,MiniLM,home with a private pool and outdoor entertaining space,2,1159158818,Blythe,0.733873,"pool, pool home this can be your vacation home, investmen o short tem rental airbnb. this inviting 3 bedroom, 2 bathroom 1 story home sits in a quiet neighborhood just minutes from the scenic colorado river, offering a perfect blend of comfort and outdoor lifestyle. step outside to a spacious backyard oasis, highlighted by a private swimming pool perfect for cooling off during hot summer days. the yard also include a covered patio area, ideal for barbecues, relaxing evenings, or enjoying views of the surrounding landscape. the home features an open concept living area with plenty of natural light, connecting the living room, dining space, and a good size kitchen. the layout is designed for both everyday living and entertaining. stay comfortable year round with central air conditioning system. the primary bedroom includes a private en suite bathroom and generous closet space, while the two additional bedrooms are ideal for family, guests, or a home office. a second full bathroom is conveniently located nearby. both bathrooms have been upgrated.this home also offers fresh interior exterior paint, new carpet. few minutes away from 10fwy"
2,MiniLM,home with a private pool and outdoor entertaining space,3,1158583082,Palmdale,0.696034,"ready for your high efficiency pool home here it is palmdale pool home with 4 bedroom and 2 bathroom. this home features triple pane windows, new tankless water heater, upgraded electrical panel, foam insulated walls, and zone heating and cooling. 2 evaporative coolers and central heat, plus gas stove. whole home water filtration system. living room has vaulted wood ceilings with large windows that create a bright open space. remodeled kitchen with new stainless steel appliances. spacious master bedroom with 2 closets 1 walk in . remodeled en suite master bathroom. 3 secondary bedrooms. remodeled hall bath with jacuzzi tub. new carpet, faux wood window covers, and paint throughout. the private backyard is a great entertaining space with 2 covered patio areas. enjoy the pebble tech in ground pool, with all upgraded plumbing and newer equipment. fully fenced for safety."
3,MiniLM,home with a private pool and outdoor entertaining space,4,1174462334,Lake Balboa,0.693828,"entertainer's dream with pool in coveted lake balboa welcome to this charming and well located home offering comfort, functionality, and the ultimate southern california lifestyle with your own private pool. step inside to a bright and inviting interior filled with natural light, featuring a functional layout that flows seamlessly through the living, dining, and kitchen areas, perfect for both everyday living and entertaining. the living space provides a warm and welcoming atmosphere, ideal for relaxing or hosting guests. the true highlight of this home is the private backyard retreat, complete with a sparkling pool, perfect for cooling off on warm days, entertaining friends and family, or simply enjoying your own outdoor oasis. the home also offers comfortable bedrooms and versatile spaces to fit your lifestyle needs, whether it's a home office, guest room, or creative space. located on a corner of this retreat also offers the abilty to covert the garage to and accessory dwelling unit. conveniently located near shopping, dining, parks, and major commuter routes, this lake balboa property offers both accessibility and lifestyle appeal. this is a fantastic opportunity to own a home with a pool in los angeles, schedule your private showing today and make it yours"
4,MiniLM,home with a private pool and outdoor entertaining space,5,1158382191,Norwalk,0.691151,"charm

In [9]:
precision_at_5 = (
    review_df.groupby("method")
    .agg(
        query_result_pairs=("_relevant", "size"),
        precision_at_5=("_relevant", "mean"),
    )
    .reset_index()
)

precision_at_5

,method,query_result_pairs,precision_at_5
0,BM25,50,0.96
1,MPNet,50,0.90
2,MiniLM,50,0.86
